# VISUALIZING THE COUNTRIES



## Proportional stacked bar chart

This is the starting point for investigating the international connections of the six Italian institutions. First, we establish the baseline geographic distribution of their citation networks. By calculating the proportional contribution of the top 10 cited and citing countries, we map the fundamental orientation of these institutions within the global scientific landscape.

This step is essential to verify whether the institutions operate within a shared international framework or if there are immediate, macro-level discrepancies in their reach.

In [20]:
import pandas as pd
import plotly.express as px
from pathlib import Path

BASE_PATH = Path("../data/citation_counts")

INSTITUTIONS = {
    "UNIBO": BASE_PATH / "UNIBO",
    "UNIMI": BASE_PATH / "UNIMI",
    "UNIPD": BASE_PATH / "UNIPD",
    "UNITO": BASE_PATH / "UNITO",
    "UPO": BASE_PATH / "UPO",
    "SNS": BASE_PATH / "SNS"
}

def load_country_data(institution, direction):
    path = INSTITUTIONS[institution] / f"citation_counts_countries_{direction}.csv"
    if path.exists():
        return pd.read_csv(path)
    return None

def plot_proportional_citations(direction, title):
    all_data = []
    
    for inst in INSTITUTIONS.keys():
        df = load_country_data(inst, direction)
        
        if df is not None:
            df['Institution'] = inst
            df = df[df['country_code'] != 'IT'] # Drop Italy
            all_data.append(df)
            
            
    if not all_data:
        print(f"No data found for direction: {direction}")
        return
        
    combined_df = pd.concat(all_data, ignore_index=True)
    
    top_10_countries = combined_df.groupby('country_name')['count'].sum().nlargest(10).index

    # Filter the dataframe to keep only those 10 countries
    final_df = combined_df[combined_df['country_name'].isin(top_10_countries)].copy()
    totals = final_df.groupby('Institution')['count'].transform('sum')
    final_df['Percentage'] = (final_df['count'] / totals) * 100

    colors_10 = [
        '#320E3B',
        '#4D1343',
        '#69184B',
        '#861D53',
        '#A3225B',
        '#BF2862',
        '#C84457',
        '#D0604D',
        '#C47D30',
        '#B7990D'
    ]

    # Create the Plotly chart
    fig = px.bar(
        final_df,
        x="Institution",
        y="Percentage",
        color="country_name",
        title=title,
        category_orders={"country_name": list(top_10_countries)}, 
        color_discrete_sequence=colors_10,
        labels={
            "Percentage": "Proportion among Top 8 (%)",
            "country_name": "Country",
            "Institution": "Institution"
        },
        hover_data={"count": True, "Percentage": ':.2f'} 
    )

    fig.update_layout(
        barmode='stack',
        template="plotly_white",
        title_font=dict(size=18, family="Arial, sans-serif"),
        hoverlabel=dict(bgcolor="white", font_size=13),
        legend=dict(
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        )
    )

    fig.show()

# Citing entities (inbound)
plot_proportional_citations(
    direction='inbound', 
    title='International Geographic Distribution of Citing Entities (Top 10)'
)

# Cited entities (outbound)
plot_proportional_citations(
    direction='outbound', 
    title='International Geographic Distribution of Cited Entities (Top 10)'
)

### Results

At a macro level, the geographic distribution of both inbound (cited) and outbound (citing) citations reveals a highly isomorphic pattern across the six Italian universities. The stacked proportional bar charts confirm that these institutions are firmly integrated into the traditional global scientific core.

Specifically, the US, France and the UK consistently emerge as the dominant hubs, collectively accounting for the vast majority of international citation flows.

However, while these charts demonstrate a shared macro-level integration, they also mask the specific insitutional behavior hidden beneath these uniform volume aggregates. To further analyze the data and deepen our exploration we use alternatives highlighting how each insitution differs from the global network and other hidden patterns.

## Heatmap

As stated before, the stacked bar charts confirm broad integration into global networks, but they mask the institution-specific specializations that differentiate the research profiles of the six Italian institutions.


To highlight these nuances, we pivot to a relative specialization analysis. By calculating the deviation of each institution from the six-university mean, we can strip away the global noise created by the dominance of major countries like the US, and instead highlight the unique geographic fingerprints of each university.

In [19]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path


BASE_PATH = Path("../data/citation_counts")
INSTITUTIONS = ["SNS", "UPO", "UNITO", "UNIMI", "UNIPD", "UNIBO"]

def plot_heatmap(direction, title, top_n=15):
    all_data = []

    # Load the data
    for inst in INSTITUTIONS:
        file_path = BASE_PATH / inst / f"citation_counts_countries_{direction}.csv"
        if file_path.exists():
            df = pd.read_csv(file_path)
            df['Institution'] = inst
            
            if 'country_code' in df.columns:
                df = df[df['country_code'] != 'IT']
                
            all_data.append(df)

    if not all_data:
        print(f"No data found for direction: {direction}")
        return

    combined_df = pd.concat(all_data, ignore_index=True)

    # Calculate true proportions before filtering countries
    totals = combined_df.groupby('Institution')['count'].transform('sum')
    combined_df['Proportion'] = (combined_df['count'] / totals) * 100

    # Identify Top N countries overall
    top_countries = combined_df.groupby('country_name')['count'].sum().nlargest(top_n).index

    # Filter data and pivot
    heatmap_df = combined_df[combined_df['country_name'].isin(top_countries)].copy()
    pivot_df = heatmap_df.pivot(index='country_name', columns='Institution', values='Proportion').fillna(0)
    pivot_df = pivot_df[INSTITUTIONS]

    # Calculate Average and Deviation
    pivot_df['Average'] = pivot_df.mean(axis=1)
    deviation_df = pivot_df[INSTITUTIONS].sub(pivot_df['Average'], axis=0)

    deviation_df['Average'] = pivot_df['Average']
    deviation_df = deviation_df.sort_values(by='Average', ascending=True) 
    
    averages = deviation_df['Average'].values
    countries = deviation_df.index.tolist()
    
    y_labels_with_avg = [f"{country} (Avg: {avg:.1f}%)" for country, avg in zip(countries, averages)]
    
    true_proportions = deviation_df[INSTITUTIONS].values + averages[:, None]
    
    deviation_df = deviation_df.drop(columns=['Average'])

    custom_colorscale = [
        [0.0, '#B7990D'], # Under-indexing
        [0.5, '#FFFFFF'], # Average
        [1.0, '#320E3B']  # Over-indexing
    ]

    vmax = np.abs(deviation_df.values).max()

    # Create the Heatmap
    fig = go.Figure(data=go.Heatmap(
        z=deviation_df.values,
        x=deviation_df.columns,
        y=y_labels_with_avg,
        customdata=true_proportions,
        colorscale=custom_colorscale,
        zmin=-vmax,
        zmax=vmax,
        zmid=0, 
        hovertemplate=(
            "<b>Institution:</b> %{x}<br>" +
            "<b>Country:</b> %{y}<br>" +
            "<b>Deviation from Avg:</b> %{z:+.1f} pp<br>" +
            "<b>True Proportion:</b> %{customdata:.1f}%<br>" + 
            "<extra></extra>" 
        ),
        colorbar=dict(title="Deviation (pp)", titleside="right"),
        xgap=1, 
        ygap=1
    ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family="Arial, sans-serif")),
        template="plotly_white",
        xaxis=dict(title="", tickfont=dict(size=12, weight="bold"), side="top"),
        yaxis=dict(title="", tickfont=dict(size=12)),
        width=850,
        height=600,
        margin=dict(t=100, l=180)
    )

    annotations = []
    for i, row in enumerate(deviation_df.values):
        for j, val in enumerate(row):
            text_color = "white" if abs(val) > (vmax * 0.6) else "black"
            annotations.append(dict(
                x=deviation_df.columns[j], 
                y=y_labels_with_avg[i],
                text=f"{val:+.1f}",
                font=dict(color=text_color, size=10),
                showarrow=False
            ))
            
    fig.update_layout(annotations=annotations)

    fig.show()

# --- Execute the Interactive Heatmaps ---
plot_heatmap(
    direction='inbound', 
    title='Relative Geographic Specialization: Inbound Citations (Top 15 Countries)'
)

plot_heatmap(
    direction='outbound', 
    title='Relative Geographic Specialization: Outbound Citations (Top 15 Countries)'
)

### Results

By calculating each university's deviation from the group meanm, the relative specialization heatmaps reveal distinct institutional behaviors that are invisible in raw volume counts.

For instance, the Scuola Normale Superiore (SNS) shows a significantly higher reliance on the US - since the resulting deviation is +2.9pp from the average. Conversely, institutions such as UPO and UNITO exhibit a deeper structural alignment with Chinese research networks.

These deviations indicate that while the Italian science core operates within a unified global hierarchy, individual geographic spheres of influence might be dictated by other aspects, such as localized strategic priorities and specific disciplinary focuses.